# Retrieval + LLM Answering (RAG)

Given an employee's question, this notebook:
1. Embeds the query
2. Retrieves the most relevant chunks from Pinecone (optionally filtered by department)
3. Builds a grounded prompt with citations
4. Calls the OpenAI chat model to produce a trustworthy, source-cited answer
5. Falls back to an honest "I don't have enough information" response when retrieval
   confidence is too low, instead of letting the model guess

This is the same logic used by `app.py` (the Streamlit chat UI).

**Requirements:** `pip install pinecone-client openai python-dotenv`


In [2]:
import os
from openai import OpenAI
from pinecone import Pinecone
from dotenv import load_dotenv

load_dotenv()

PINECONE_INDEX_NAME = "ironstore-enterprise-knowledge-base"
EMBEDDING_MODEL = "text-embedding-3-large"
CHAT_MODEL = "gpt-4o-mini"

TOP_K = 6
SCORE_THRESHOLD = 0.72  # below this top-match score, we don't trust the retrieval enough to answer

openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index = pc.Index(PINECONE_INDEX_NAME)


## 1. Embed the query

In [3]:
def embed_query(query: str):
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=[query])
    return response.data[0].embedding


## 2. Retrieve relevant chunks

If `department` is given, search only that namespace. Otherwise search across all
department namespaces and merge results by score.


In [4]:
def retrieve(query: str, top_k: int = TOP_K, department=None):
    query_vector = embed_query(query)

    if department:
        namespaces = [department]
    else:
        stats = index.describe_index_stats()
        namespaces = list(stats.get("namespaces", {}).keys())

    all_matches = []
    for ns in namespaces:
        result = index.query(vector=query_vector, top_k=top_k, namespace=ns, include_metadata=True)
        all_matches.extend(result["matches"])

    all_matches.sort(key=lambda m: m["score"], reverse=True)
    return all_matches[:top_k]


## 3. Format context + citations, and define the system prompt

In [5]:
def format_context(matches):
    """Build a numbered context block for the LLM, and a parallel list of source citations."""
    context_blocks = []
    sources = []
    for i, m in enumerate(matches, start=1):
        meta = m["metadata"]
        context_blocks.append(
            f"[{i}] Department: {meta['department']} | Document: {meta['document_name']} "
            f"| Section: {meta['section_title']} | Pages: {meta['start_page']}-{meta['end_page']}\n"
            f"{meta['text']}"
        )
        sources.append({
            "ref": i,
            "department": meta["department"],
            "document_name": meta["document_name"],
            "section_title": meta["section_title"],
            "start_page": meta["start_page"],
            "end_page": meta["end_page"],
            "score": round(m["score"], 3),
        })
    return "\n\n---\n\n".join(context_blocks), sources


SYSTEM_PROMPT = """You are an internal enterprise assistant. Answer employee questions \
using ONLY the provided context excerpts from internal company documents.

Rules:
1. Base your answer strictly on the given context. Do not use outside knowledge or make assumptions.
2. Always cite which excerpt(s) you used, like this: (Source [1], [3]).
3. If the context does not contain enough information to answer confidently, say clearly: \
"I don't have enough information in the internal documents to answer this confidently," \
and suggest which department or document the employee might check with instead.
4. Be concise, clear, and professional. Explain procedures step by step when relevant.
5. Never fabricate policies, numbers, or procedures that are not present in the context.
"""


## 4. Full RAG pipeline

If the top retrieval score is below `SCORE_THRESHOLD`, we skip the LLM call entirely and
return an honest "not confident" response — this keeps the assistant trustworthy and
avoids the model hallucinating an answer from weak or irrelevant context.


In [9]:
def generate_answer(query: str, department=None, top_k: int = TOP_K):
    matches = retrieve(query, top_k=top_k, department=department)

    # if not matches or matches[0]["score"] < SCORE_THRESHOLD:
    #     return {
    #         "answer": (
    #             "I couldn't find sufficiently relevant information in the internal documents "
    #             "to answer this confidently. You may want to check with the relevant department "
    #             "directly or try rephrasing your question."
    #         ),
    #         "sources": [],
    #         "confident": False,
    #     }

    context, sources = format_context(matches)
    user_prompt = f"Context excerpts:\n\n{context}\n\nEmployee question: {query}"

    response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": sources,
        "confident": True,
    }


## 5. Try it out

In [10]:
result = generate_answer("What is the process for requesting annual leave?")

print(result["answer"])
print("\nSources:")
for s in result["sources"]:
    print(f"  [{s['ref']}] {s['department']} / {s['document_name']} \u2014 {s['section_title']} "
          f"(pages {s['start_page']}-{s['end_page']}, score {s['score']})")


To request annual leave, employees should follow these steps:

1. **Submit Request**: Use PeopleHub to submit your annual leave request.
2. **Timing**: Ensure you submit your request at least:
   - 10 calendar days in advance for leave of one to five working days.
   - 20 calendar days in advance for leave exceeding five working days.
   - As early as possible during peak periods (July–August, year-end, and major local holidays).
3. **Await Manager Response**: Your manager will respond within five business days. Approval will depend on operational coverage, customer commitments, fulfilment schedules, and previously approved leave.
4. **Consider Alternative Dates**: If multiple employees request the same period, managers may propose alternative dates.
5. **Do Not Book Travel Until Approved**: Avoid booking non-refundable travel until your leave is officially approved.

Remember, leave is not confirmed until it has been approved by your line manager (Source [1], [2], [5]).

Sources:
  [1

In [12]:
# Example: restrict to a single department
result = generate_answer("How do I submit an expense report?", department="finance")
print(result["answer"])


I don't have enough information in the internal documents to answer this confidently. I suggest checking with the Finance department or the company’s expense reporting policy document for detailed procedures on submitting an expense report.
